<a href="https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/01_preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/skyexry/urban-mobility-forecast

fatal: destination path 'urban-mobility-forecast' already exists and is not an empty directory.


In [20]:
import sys
import subprocess
from tqdm import tqdm
import glob
import dask.dataframe as dd
from dask.diagnostics import ProgressBar
import pandas as pd
import os


In [3]:
sys.path.append('/content/urban-mobility-forecast')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
urls = [
    "https://s3.amazonaws.com/tripdata/202401-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202402-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202403-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202404-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202405-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202406-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202407-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202408-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202409-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202410-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202411-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202412-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202501-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202502-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202503-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202504-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202505-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202506-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202507-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202508-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202509-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202510-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202511-citibike-tripdata.zip",
    "https://s3.amazonaws.com/tripdata/202512-citibike-tripdata.zip",
]

for url in tqdm(urls, desc="Downloading"):
    filename = url.split("/")[-1]
    subprocess.run(["wget", "-q", url, "-O", f"/content/{filename}"])

In [10]:
zip_files = sorted(glob.glob("/content/*.zip"))

for f in tqdm(zip_files, desc="Unzipping"):
    subprocess.run(["unzip", "-q", "-o", f, "-d", "/content/"],
                   capture_output=True)

Unzipping: 100%|██████████| 24/24 [03:09<00:00,  7.89s/it]


In [15]:
# Find all citibike CSV files recursively
csv_files = sorted(glob.glob("/content/**/*citibike*.csv", recursive=True))
print(f"Found {len(csv_files)} CSV files")

Found 144 CSV files


In [16]:
# Check column names across all CSV files
all_columns = {}
for f in tqdm(csv_files, desc="Checking columns"):
    cols = tuple(pd.read_csv(f, nrows=0).columns.tolist())
    all_columns[f] = cols

# Find unique column sets
unique_col_sets = set(all_columns.values())
print(f"Number of unique column schemas: {len(unique_col_sets)}")
for col_set in unique_col_sets:
    print(col_set)

Checking columns: 100%|██████████| 144/144 [00:05<00:00, 24.39it/s]

Number of unique column schemas: 1
('ride_id', 'rideable_type', 'started_at', 'ended_at', 'start_station_name', 'start_station_id', 'end_station_name', 'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng', 'member_casual')


In [17]:
# Load only required columns to save memory
df = dd.read_csv(
    csv_files,
    usecols=['ride_id', 'started_at', 'start_station_id', 'start_station_name', 'start_lat', 'start_lng'],
    dtype={'start_station_id': 'object', 'start_lat': 'float64', 'start_lng': 'float64'}
)

In [18]:
# Floor datetime to hourly granularity
df['hour'] = dd.to_datetime(df['started_at']).dt.floor('h')

In [19]:
# Count departures per station per hour
hourly = (
    df.groupby(['start_station_id', 'start_station_name', 'start_lat', 'start_lng', 'hour'])
    .ride_id.count()
    .reset_index()
    .rename(columns={'ride_id': 'demand'})
)

In [21]:
# Trigger computation with progress bar
with ProgressBar():
    result = hourly.compute()

print(f"Done. Shape: {result.shape}")
print(result.head())

[########################################] | 100% Completed | 18m 53s
Done. Shape: (19535002, 6)
  start_station_id           start_station_name  start_lat  start_lng  \
0          2898.01  Cortelyou Rd & Stratford Rd  40.639660 -73.968070   
1          2898.01  Cortelyou Rd & Stratford Rd  40.639660 -73.968070   
2          3056.05               12 Ave & 36 St  40.643546 -73.986418   
3          3090.06    Church Ave & McDonald Ave  40.642809 -73.979239   
4          3117.05                46 St & 7 Ave  40.644743 -74.003754   

                 hour  demand  
0 2025-04-04 10:00:00       8  
1 2025-04-05 10:00:00       4  
2 2025-04-03 06:00:00       2  
3 2025-04-12 13:00:00      10  
4 2025-04-13 15:00:00       6  


In [22]:
# Create output directory in Google Drive
os.makedirs("/content/drive/MyDrive/citibike", exist_ok=True)

In [34]:
# Save as parquet
result.to_parquet("/content/drive/MyDrive/citibike/hourly_demand_full.parquet", index=False)
print("Saved to Google Drive")

Saved to Google Drive


In [35]:
# Filter stations and save final parquet
# Step 1: rank stations by total demand, keep top 500
station_demand = (result.groupby('start_station_id')['demand']
                  .sum()
                  .reset_index()
                  .rename(columns={'demand': 'total_demand'})
                  .sort_values('total_demand', ascending=False))

top500_ids = station_demand.head(500)['start_station_id'].tolist()
result_500 = result[result['start_station_id'].isin(top500_ids)]

In [36]:
# Step 2: filter stations with >= 70% time coverage
total_hours = result_500['hour'].nunique()
station_coverage = (result_500.groupby('start_station_id')['hour']
                    .nunique()
                    .reset_index()
                    .rename(columns={'hour': 'hours_count'}))

valid_ids = station_coverage[
    station_coverage['hours_count'] >= total_hours * 0.7
]['start_station_id'].tolist()

In [37]:
# Step 3: trim date range and finalize
result_final = (result_500[result_500['start_station_id'].isin(valid_ids)]
                .query("hour >= '2024-01-01' and hour <= '2025-12-31 23:00:00'")
                .copy())

print(f"Final stations : {result_final['start_station_id'].nunique()}")
print(f"Final shape    : {result_final.shape}")
print(f"Date range     : {result_final['hour'].min()} to {result_final['hour'].max()}")

Final stations : 438
Final shape    : (5956935, 6)
Date range     : 2024-01-01 00:00:00 to 2025-12-31 23:00:00


In [38]:
# Step 4: save once to Google Drive
os.makedirs("/content/drive/MyDrive/citibike", exist_ok=True)
result_final.to_parquet("/content/drive/MyDrive/citibike/hourly_demand_final.parquet", index=False)
print("Saved to Google Drive")

Saved to Google Drive
